# Actividad autónoma — Estructuras de Datos
## Unidad 1 · Tema 2: Normalización y procesamiento de datos para gestión logística

**Nombre:** Joseph Damian Pardo Gualan

**Fecha:** 30 - 10 - 2026

**Semestre:** 2do ciclo paralelo "B"

**Objetivo:** convertir 10 registros de paquetes escritos de manera inconsistente en una **lista de diccionarios**, calcular impuestos y totales, realizar un análisis y añadir una función original.

### Importar diccionarios


In [ ]:
import re  # Importamos esta libreria para identifica patrones en el texto.
from math import isclose  # La usaremos más adelante para comparar resultados decimales.

### 1. Dataset original

In [12]:
# Creamos una lista que contiene los 10 textos originales de la actividad.
data_set = [
    "ID:TRK901;   PRODUCTO: laptop_MACBOOK_PRO           ;   ORIGEN:China; PRECIO:1200.5; TASA:0.15",  # Registro 1: usa ; y :.
    "id-TRK902, producto- camara_SONY, origen-Japon, precio-850.0, tasa-0.10",  # Registro 2: usa , y -.
    "ID:TRK903;       PRODUCTO:consola_PLAYSTATION;           ORIGEN:USA; PRECIO:499.9; TASA:0.12####",  # Registro 3: tiene # al final.
    "    id:TRK904 - producto: monitor_DELL - origen:Corea - precio:350.0 - tasa:0.08",  # Registro 4: mezcla : y ' - '.
    "ID:TRK905;      PRODUCTO:teclado_MECANICO;              ORIGEN:China; PRECIO:120.0; TASA:0.15",  # Registro 5: espacios sobrantes.
    "id-TRK906, producto- audifonos_BOSE, origen-USA, precio-299.0, tasa-0.12????",  # Registro 6: tiene ? al final.
    "ID:TRK907;    PRODUCTO: tablet_SAMSUNG          ;   ORIGEN:Vietnam; PRECIO:650.0; TASA:0.10",  # Registro 7: espacios sobrantes.
    "id:TRK908 - producto: disco_DURO - origen:China - precio:90.0 - tasa:0.15",  # Registro 8: usa ' - '.
    "ID:TRK909; PRODUCTO:reloj_GARMIN; ORIGEN:Suiza; PRECIO:450.0; TASA:0.05!!!!",  # Registro 9: tiene ! al final.
    "id-TRK910, producto- microfono_SHURE, origen-Mexico, precio-210.0, tasa-0.07",  # Registro 10: usa , y -.
]

print(f"Cantidad de registros originales: {len(data_set)}")  # Verificamos que haya 10 registros.


Cantidad de registros originales: 10


## Parte 1. Normalizar y estructurar los registros

El objetivo es transformar esa lista de strings en una lista de registros (diccionarios) perfectamente estructurada.
1. Notarás que algunos registros usan; otros, y otros -. Debes normalizarlos antes de separar los datos.
2. Cada registro debe tener exactamente estas llaves

- id_rastreo: Mantener el código alfanumérico (ej: "TRK901").
- item: Nombre del producto en formato título, reemplazando guiones bajos por espacios.
- pais_origen: El nombre del país en MAYÚSCULAS.
- impuesto_valor: Calcular el valor del impuesto multiplicando PRECIO por TASA.
- total_pagar: El PRECIO original más el impuesto_valor.

3. Eliminar los caracteres especiales al final (#, ?, !) y cualquier espacio en blanco en los extremos.

In [ ]:
def convertir_paquete(cadena):
    # 1) Eliminamos espacios al inicio y al final de la cadena recibida.
    texto = cadena.strip()

    # 2) Quitamos uno o varios símbolos #, ? o ! que estén al final del registro.
    texto = texto.rstrip("#?!")

    # 3) Volvemos a eliminar espacios sobrantes que pudieran quedar al final.
    texto = texto.strip()

    # 4) Buscamos guiones con espacios a ambos lados: estos separan campos completos.
    #    La expresión r"\s+-\s+" significa: espacios + guion + espacios.
    texto = re.sub(r"\s+-\s+", ";", texto)

    # 5) Convertimos las comas separadoras en punto y coma para unificar el formato.
    texto = texto.replace(",", ";")

    # 6) Dividimos el registro en una lista de fragmentos usando el separador común (;).
    partes = texto.split(";")

    # 7) Creamos un diccionario temporal con las claves originales: id, producto, origen, etc.
    campos = {}

    # 8) Recorremos cada fragmento, por ejemplo: ' PRODUCTO: laptop_MACBOOK_PRO '.
    for parte in partes:
        parte = parte.strip()  # Quitamos espacios innecesarios de este fragmento.

        if not parte:  # Si una parte está vacía, no intentamos procesarla.
            continue  # Saltamos a la siguiente vuelta del ciclo.

        # 9) Separamos SOLO en la primera aparición de ':' o '-' (maxsplit=1).
        #    Ejemplos: 'ID:TRK901' y 'id-TRK902' producen dos elementos.
        clave_y_valor = re.split(r"\s*[:-]\s*", parte, maxsplit=1)

        # 10) Comprobamos que realmente existan una clave y su valor.
        if len(clave_y_valor) != 2:
            raise ValueError(f"Campo con formato incorrecto: {parte!r}")

        # 11) Guardamos cada elemento en una variable con un nombre descriptivo.
        clave, valor = clave_y_valor

        # 12) Convertimos la clave a minúsculas para tratar igual 'ID', 'Id' e 'id'.
        clave = clave.strip().lower()

        # 13) Limpiamos los espacios del valor antes de almacenarlo.
        valor = valor.strip()

        # 14) Guardamos la información normalizada en el diccionario temporal.
        campos[clave] = valor

    # 15) Validamos que estén presentes los cinco campos del dataset original.
    obligatorios = ("id", "producto", "origen", "precio", "tasa")
    for nombre_campo in obligatorios:
        if nombre_campo not in campos:
            raise ValueError(f"Falta el campo {nombre_campo!r} en el registro: {cadena!r}")

    # 16) Convertimos el precio, que ahora es texto, en un número decimal (float).
    precio = float(campos["precio"])

    # 17) Convertimos la tasa a float para poder multiplicarla por el precio.
    tasa = float(campos["tasa"])

    # 18) Calculamos el impuesto: precio del producto × tasa asignada.
    impuesto = precio * tasa

    # 19) Calculamos el total de pago: precio original + impuesto.
    total = precio + impuesto

    # 20) Creamos el diccionario final con EXACTAMENTE las cinco claves solicitadas.
    registro = {
        "id_rastreo": campos["id"],  # Conservamos el código alfanumérico del paquete.
        "item": campos["producto"].replace("_", " ").title(),  # Quitamos _ y usamos formato título.
        "pais_origen": campos["origen"].upper(),  # Convertimos el nombre del país a MAYÚSCULAS.
        "impuesto_valor": impuesto,  # Guardamos el resultado numérico del impuesto.
        "total_pagar": total,  # Guardamos el costo total, incluido el impuesto.
    }

    # 21) Devolvemos el diccionario final y, aparte, el precio antes de impuestos.
    #     Mantenemos el precio aparte porque NO figura entre las cinco claves requeridas.
    return registro, precio

### Procesar los 10 paquetes

In [5]:
# Almacenará únicamente los diccionarios finales con las cinco claves pedidas.
registros = []

# Almacenará los precios sin impuestos, necesarios para calcular el promedio.
precios_originales = []

# Recorremos todas las cadenas del dataset, una por una.
for cadena in data_set:
    registro, precio = convertir_paquete(cadena)  # Limpiamos y transformamos el texto.
    registros.append(registro)  # Agregamos el diccionario final a la lista de registros.
    precios_originales.append(precio)  # Guardamos el precio original en la lista auxiliar.

# Comprobamos cuántos paquetes se han procesado correctamente.
print(f"Paquetes procesados: {len(registros)}")

# Recorremos la lista para verificar de manera visible la estructura de los 10 paquetes.
for numero, registro in enumerate(registros, start=1):
    print(f"Paquete {numero}: {registro}")  # enumerate() numera desde 1 para facilitar la lectura.

Paquetes procesados: 10
Paquete 1: {'id_rastreo': 'TRK901', 'item': 'Laptop Macbook Pro', 'pais_origen': 'CHINA', 'impuesto_valor': 180.075, 'total_pagar': 1380.575}
Paquete 2: {'id_rastreo': 'TRK902', 'item': 'Camara Sony', 'pais_origen': 'JAPON', 'impuesto_valor': 85.0, 'total_pagar': 935.0}
Paquete 3: {'id_rastreo': 'TRK903', 'item': 'Consola Playstation', 'pais_origen': 'USA', 'impuesto_valor': 59.98799999999999, 'total_pagar': 559.8879999999999}
Paquete 4: {'id_rastreo': 'TRK904', 'item': 'Monitor Dell', 'pais_origen': 'COREA', 'impuesto_valor': 28.0, 'total_pagar': 378.0}
Paquete 5: {'id_rastreo': 'TRK905', 'item': 'Teclado Mecanico', 'pais_origen': 'CHINA', 'impuesto_valor': 18.0, 'total_pagar': 138.0}
Paquete 6: {'id_rastreo': 'TRK906', 'item': 'Audifonos Bose', 'pais_origen': 'USA', 'impuesto_valor': 35.879999999999995, 'total_pagar': 334.88}
Paquete 7: {'id_rastreo': 'TRK907', 'item': 'Tablet Samsung', 'pais_origen': 'VIETNAM', 'impuesto_valor': 65.0, 'total_pagar': 715.0}
Pa

## Parte 2. Operaciones de análisis
**Instrucciones:**
Una vez que la lista de registros esté lista, el programa debe:
- Mostrar un resumen con: ID, Producto y el Total a Pagar.
- Identificar y mostrar el producto que pagó el impuesto más alto.
- Calcular el valor promedio de todos los productos antes de impuestos.

In [ ]:
# Mostrar un resumen con: ID, Producto y el Total a Pagar.
print("=" * 72)
print(f"{'ID':<10} {'PRODUCTO':<30} {'TOTAL A PAGAR (USD)':>22}")
print("=" * 72)

# Recorremos cada diccionario estructurado para imprimir un resumen por paquete.
for registro in registros:
    codigo = registro["id_rastreo"]  # Extraemos la identificación del paquete.
    producto = registro["item"]  # Extraemos el nombre normalizado del producto.
    total = registro["total_pagar"]  # Extraemos el importe final calculado.

    # <10 alinea el ID a la izquierda; <30 alinea el producto; >22 alinea el total a la derecha.
    # :.2f muestra solo dos decimales en pantalla, sin cambiar el valor guardado.
    print(f"{codigo:<10} {producto:<30} {total:>22.2f}")

print("=" * 72)  # Cerramos visualmente la tabla.

ID         PRODUCTO                          TOTAL A PAGAR (USD)
TRK901     Laptop Macbook Pro                            1380.58
TRK902     Camara Sony                                    935.00
TRK903     Consola Playstation                            559.89
TRK904     Monitor Dell                                   378.00
TRK905     Teclado Mecanico                               138.00
TRK906     Audifonos Bose                                 334.88
TRK907     Tablet Samsung                                 715.00
TRK908     Disco Duro                                     103.50
TRK909     Reloj Garmin                                   472.50
TRK910     Microfono Shure                                224.70


In [ ]:
# Identificar y mostrar el producto que pagó el impuesto más alto.
# La función max() devuelve el diccionario cuyo impuesto_valor sea el mayor.
# El argumento key indica qué valor del diccionario se debe comparar.
paquete_impuesto_maximo = max(registros, key=lambda r: r["impuesto_valor"])

# Mostramos los datos principales del paquete ganador.
print("PRODUCTO CON EL IMPUESTO MÁS ALTO")
print(f"ID: {paquete_impuesto_maximo['id_rastreo']}")
print(f"Producto: {paquete_impuesto_maximo['item']}")
print(f"País: {paquete_impuesto_maximo['pais_origen']}")
print(f"Impuesto: ${paquete_impuesto_maximo['impuesto_valor']:.2f}")

PRODUCTO CON EL IMPUESTO MÁS ALTO
ID: TRK901
Producto: Laptop Macbook Pro
País: CHINA
Impuesto: $180.07


In [ ]:
# Calcular el valor promedio de todos los productos antes de impuestos.
# sum() suma los precios originales guardados en la lista auxiliar.
suma_precios = sum(precios_originales)

# len() cuenta cuántos precios existen para evitar usar una cantidad fija.
cantidad_precios = len(precios_originales)

# Dividimos la suma entre el número de paquetes para obtener el promedio aritmético.
promedio_antes_impuestos = suma_precios / cantidad_precios

# Mostramos la suma, la cantidad y el promedio, con dos decimales para lectura.
print(f"Suma de precios antes de impuestos: ${suma_precios:.2f}")
print(f"Cantidad de paquetes: {cantidad_precios}")
print(f"Promedio antes de impuestos: ${promedio_antes_impuestos:.2f}")

Suma de precios antes de impuestos: $4719.40
Cantidad de paquetes: 10
Promedio antes de impuestos: $471.94


## Parte 3. Función personalizada ¿Qué país aporta más impuestos?

**Propuesta adicional:** crear un resumen de impuestos agrupados por país de origen. Esta funcion permite conocer cuántos paquetes provienen de cada país, cuánto impuesto aportan y qué proporción representan de toda la recaudación. Es una forma de identificar dónde se concentra el valor de los tributos.


In [9]:
def auditoria_por_pais(lista_registros):
    """Agrupa impuestos por país y devuelve una lista de resúmenes ordenada de mayor a menor."""

    por_pais = {}  # Diccionario temporal: cada país será una clave.

    # Recorremos todos los registros que se hayan procesado.
    for registro in lista_registros:
        pais = registro["pais_origen"]  # Identificamos de qué país viene el paquete.
        impuesto = registro["impuesto_valor"]  # Consultamos el impuesto del paquete.

        # Si el país todavía no aparece, creamos sus contadores desde cero.
        if pais not in por_pais:
            por_pais[pais] = {
                "cantidad": 0,  # Número de paquetes recibidos desde ese país.
                "impuestos": 0.0,  # Suma de impuestos aportados por esos paquetes.
            }

        # Contamos un paquete adicional para el país correspondiente.
        por_pais[pais]["cantidad"] += 1

        # Acumulamos el impuesto del paquete en el total de ese país.
        por_pais[pais]["impuestos"] += impuesto

    # Sumamos todos los impuestos para hallar la recaudación total.
    recaudacion_total = sum(datos["impuestos"] for datos in por_pais.values())

    resultados = []  # Aquí guardaremos una fila de resumen por cada país.

    # items() entrega el nombre del país y su diccionario con los acumulados.
    for pais, datos in por_pais.items():
        # Calculamos el porcentaje que aporta cada país a toda la recaudación.
        # Si la recaudación fuera cero, evitamos una división entre cero.
        porcentaje = (datos["impuestos"] / recaudacion_total * 100) if recaudacion_total else 0

        # Creamos una fila de resultados que solo se usará en este reporte.
        resultados.append({
            "pais": pais,  # País de procedencia de los paquetes.
            "cantidad": datos["cantidad"],  # Número de paquetes desde ese país.
            "impuestos": datos["impuestos"],  # Impuestos acumulados del país.
            "porcentaje": porcentaje,  # Participación del país en el total.
        })

    # Ordenamos los países desde el que aporta más impuestos hasta el que aporta menos.
    resultados.sort(key=lambda fila: fila["impuestos"], reverse=True)

    return resultados, recaudacion_total  # Devolvemos el detalle y el total global.


# Ejecutamos la nueva función con nuestra lista original de diccionarios.
reporte_paises, total_impuestos = auditoria_por_pais(registros)

# Mostramos el resumen calculado por país.
print("AUDITORÍA DE IMPUESTOS POR PAÍS DE ORIGEN")
print("-" * 65)
print(f"{'PAÍS':<15} {'PAQUETES':>10} {'IMPUESTOS USD':>18} {'PARTICIPACIÓN':>17}")
print("-" * 65)

# Imprimimos una fila por país con sus cantidades e impuestos acumulados.
for fila in reporte_paises:
    print(f"{fila['pais']:<15} {fila['cantidad']:>10} {fila['impuestos']:>18.2f} {fila['porcentaje']:>16.1f}%")

print("-" * 65)
print(f"RECAUDACIÓN TOTAL: ${total_impuestos:.2f}")

# Como las filas están ordenadas de mayor a menor, la primera es el país líder.
if reporte_paises:
    print(f"País que más aporta impuestos: {reporte_paises[0]['pais']}")

AUDITORÍA DE IMPUESTOS POR PAÍS DE ORIGEN
-----------------------------------------------------------------
PAÍS              PAQUETES      IMPUESTOS USD     PARTICIPACIÓN
-----------------------------------------------------------------
CHINA                    3             211.57             40.5%
USA                      2              95.87             18.3%
JAPON                    1              85.00             16.3%
VIETNAM                  1              65.00             12.4%
COREA                    1              28.00              5.4%
SUIZA                    1              22.50              4.3%
MEXICO                   1              14.70              2.8%
-----------------------------------------------------------------
RECAUDACIÓN TOTAL: $522.64
País que más aporta impuestos: CHINA
